In [0]:
DROP TABLE IF EXISTS sandbox.z_jungryo_lee.agg_buzzmetrix_model_category_v1;
CREATE TABLE IF NOT EXISTS sandbox.z_jungryo_lee.agg_buzzmetrix_model_category_v1 ( 

-- 0) 공통 필터를 먼저 적용한 로우 집합 (FHD/HD는 아직 제거하지 않음: 유실 방지)
WITH base AS (
  SELECT
    country,
    year,
    CASE WHEN brand_name = 'SS' THEN 'Samsung' ELSE brand_name END AS brand_name,
    model,
    device_type,
    cate_1_depth,
    cate_2_depth,
    sc_measurement
  FROM kic_data_ods.buzzmetrix.buzzmetrix
  WHERE 1=1
    AND cate_1_depth NOT IN ('***장소', '***컨텐츠')
    AND country IN ('United States', 'United Kingdom', 'Korea', 'Germany', 'Brazil')
    AND year >= 2022      
    AND brand_name IN ('Samsung', 'LG', 'SS', 'TCL')
),

-- 1) device_type_up 중복된 model 추출 및 대표 device_type_up 
dupe_model_type as (
  SELECT
  model,
  CASE
    -- 1) OLED
    WHEN UPPER(model)        RLIKE 'OLED'
      OR UPPER(model)        RLIKE 'S9[0-9]B'                        -- 예: S95B
      THEN 'OLED'

    -- 2) QLED / QNED / NanoCell (Q 계열)
    WHEN UPPER(model)        RLIKE 'QLED'
      THEN 'QLED'
    WHEN UPPER(model)        RLIKE 'QNED'
      THEN 'QNED'
    WHEN UPPER(model)        RLIKE 'NANO|NANOCELL'
      THEN 'NANOCELL'
    WHEN UPPER(model)        RLIKE '\\bQ[0-9]{2}[A-Z]?\\b'           -- 예: Q60B/C
      OR UPPER(model)        RLIKE 'QN[0-9]{2,3}[A-Z]?'
      OR UPPER(model)        RLIKE 'Q[0-9]{3}G'
      THEN 'QLED'

    -- 3) UHD (일반 4K)
    WHEN UPPER(model)        RLIKE '\\b(UQ|UP|UX)[0-9]+'
      OR UPPER(model)        RLIKE '\\b(TU|AU|BU|CU|DU)[0-9]+'
      THEN 'UHD'
    ELSE 'FHD'
      END as resolved_type_raw
  FROM base
  WHERE 1=1
  group by model having count(distinct device_type) > 1
), 

filtered_data as (
  SELECT b.*
    , case
        when d.resolved_type_raw is not null then d.resolved_type_raw 
        else b.device_type 
      end as device_type_up 
  FROM BASE b
  LEFT JOIN dupe_model_type d Using(model)
), 

data as (
  SELECT
    `country`,
    `year`,
    device_type_up, 
    case when `brand_name`='SS' then 'Samsung' else `brand_name` end as brand_name,
    `model`,
    case when device_type_up = 'OLED' then 'OLED'
        when device_type_up = 'UHD' then 'UHD'
        else 'QLED/QNED/Nanocell'
    end as d_type, 
    cate_1_depth as category, 
    sc_measurement
  FROM
    filtered_data
  WHERE 1=1
    AND `cate_2_depth` IS NOT NULL
    AND device_type_up not like '%FHD%'

  union all

  SELECT
    `country`,
    `year`,
    device_type_up, 
    case 
      when `brand_name`='SS' then 'Samsung' else `brand_name` end as brand_name,
    `model`,
    case
      when device_type_up = 'OLED' then 'OLED'
      when device_type_up = 'UHD' then 'UHD'
      else 'QLED/QNED/Nanocell'
    end as d_type, 
    cate_2_depth as category, 
    sc_measurement
  FROM
    filtered_data
  WHERE 1=1
    AND `cate_1_depth` = '07. 스마트 사용성'   
    AND device_type_up not like '%FHD%'
    AND `cate_2_depth` IS NOT NULL
)

select country, year, brand_name, model, d_type, category,
  count(sc_measurement) total_count,
  avg(sc_measurement) avg_sc,
  count(case when sc_measurement = 1 then sc_measurement else null end) positive,
  count(case when sc_measurement = 0 then sc_measurement else null end) neutral,
  count(case when sc_measurement = -1 then sc_measurement else null end) negative

from data

group by country, year, brand_name, model, d_type, category
);

select *
from sandbox.z_jungryo_lee.agg_buzzmetrix_model_category_v1
WHERE 1=1 
  --  and model like '%LQ%'